# MassDOT Regional Data Cleaner for Mass Crash Map
# (Expected Runtime: 2-5 minutes)

Output in /data folder:

-boston-metro2.csv - contains MassDOT crash records for specified regions, formatted for Mass Crash Map

# Importing all libraries

In [1]:
# !pip install googlemaps
# uncomment the line above if we decide to pay for GOOGLE CLOUD, an estimate of the cost is below $200
# GOOGLE CLOUD can be used to geocode 16911 crashes that have addresses but not lon and lat
# more information on how to do this can be found here: https://tinyurl.com/mr42hkxw
!pip install pandas
!pip install numpy
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


# Collecting MassDOT crash data for specific regions and years (update in 2027)

In [2]:
all_features = []

#Inserted years individually because "2023v" naming issue in MassDOT crash server
#https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT

years = ('2022', '2023v','2024','2025','2026')

#EDIT Regional Planning Area (RPA) acronyms in 'regions' below
#boston-metro = ['MAPC']
#central = ['MRPC','CMRPC']
#northeast = ['NMCOG','MVPC']
#western = ['BRPC','FRCOG','PVPC']
#southeast = ['CCC','MVC','NRPEDC','OCPC','SRPEDD']

regions = ['MAPC']
region_str = ", ".join([f"'{c}'" for c in regions])

for year in years:
   
    base_url = f"https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT/MASSDOT_ODP_OPEN_{year}/FeatureServer/0/query"

    
    params = {
        "where": f"RPA_ABBR IN ({region_str})",
        "outFields": "*",
        "outSR": "4326",
        "f": "json",
        "returnGeometry": "true",
        "resultOffset": 0,
        "resultRecordCount": 2000
    }

    while True:
        
        response = requests.get(base_url, params=params)
        data = response.json()
        features = data.get("features", [])
        
        if not features:
            break
        
        all_features.extend(features)
        params["resultOffset"] += params["resultRecordCount"]

records = [f["attributes"] for f in all_features]

df = pd.DataFrame(records)
print("Shape:", df.shape)
df.head()

Shape: (219362, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATE_TEXT,CRASH_TIME_2,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,5051236,EVERETT,01 01 2022,11:34 AM,1.641037e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No injury,2,...,NaN,Major Collector,4742067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5051675,EVERETT,01 03 2022,3:12 AM,1.641180e+12,03:00AM to 03:59AM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,...,NaN,Principal Arterial - Other,4742121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5051674,EVERETT,01 02 2022,10:17 PM,1.641162e+12,10:00PM to 10:59PM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,...,NaN,Local,4742122,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5051661,DEDHAM,01 02 2022,12:36 PM,1.641127e+12,12:00PM to 12:59PM,Closed,Unknown,Not reported,2,...,NaN,Local,4742135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5051578,BRAINTREE,01 02 2022,11:49 AM,1.641124e+12,11:00AM to 11:59AM,Closed,Non-fatal injury,Possible Injury (C),3,...,NaN,Minor Arterial,4742178,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#CRASH_TIME_2 & CRASH_DATE_TEXT columns are of type str
#LAT & LON are of type float64
#The following code ensures time and date are of type datetime and lat and lon are numeric
df["CRASH_TIME"] = pd.to_datetime(df["CRASH_TIME_2"], format="%I:%M %p", errors="coerce").dt.time
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE_TEXT"], errors="coerce")
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df = df.drop(columns=["CRASH_DATE_TEXT", "CRASH_TIME_2"])
mass_crashes = df
print("Shape of MASSDOT dataset:", mass_crashes.shape)
mass_crashes.head()

Shape of MASSDOT dataset: (219362, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,NUMB_NONFATAL_INJR,NUMB_FATAL_INJR,...,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL,CRASH_TIME,CRASH_DATE
0,5051236,EVERETT,1.641037e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No injury,2,0,0,...,4742067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11:34:00,2022-01-01
1,5051675,EVERETT,1.641180e+12,03:00AM to 03:59AM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,2,0,...,4742121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03:12:00,2022-01-03
2,5051674,EVERETT,1.641162e+12,10:00PM to 10:59PM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,2,0,...,4742122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22:17:00,2022-01-02
3,5051661,DEDHAM,1.641127e+12,12:00PM to 12:59PM,Closed,Unknown,Not reported,2,0,0,...,4742135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12:36:00,2022-01-02
4,5051578,BRAINTREE,1.641124e+12,11:00AM to 11:59AM,Closed,Non-fatal injury,Possible Injury (C),3,6,0,...,4742178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11:49:00,2022-01-02


# Standardizing data and Coding vulnerable roadway user types

Context: The challenge is to group different types of vulnerable roadway users that may not uniformly appear in the MassDOT crash data, or may be confused by ambiguous terminology, because info is drawn from reports filled out by different police officers across various departments, with varying levels of quality. 

Our goal is to condense various terms into three large categories for the Mass. Crash Map:
TODO MORE HERE

- "Pedestrian" or "Bicyclist" or several others in "Non-Motorist Type" column 1
- "Collision with pedestrian" or "Collision with cyclist" or several others in "Most Harmful Event" column 2
However, sometimes the term "Bicyclist" will NOT appear in column 1, while "Collision with cyclist" will appear in column 2

**TODO: UPDATE with improved logic for 1 values, and insert blanks instead of 0s for false** 

**The following code creates new columns:**
* The PEDESTRIAN column has a value of 1 if a pedestrian was involved in a crash, 0 otherwise
* The CYCLIST column has a value of 1 if a cyclist OR micromobility user was involved in a crash, 0 otherwise
* The OTHER column has a value of 1 if another vulnerable user was involved in a crash, 0 otherwise

**Pedestrian-type terms (based on speed similar to walking):** 
* Pedestrian
* Electric Personal Assistive Mobility Device User
* Non-Motorized Wheelchair User
* Emergency Responder - Outside of vehicle
* Roadway Worker - Outside of vehicle
* Utility Worker - Outside of vehicle

**Cyclist-type terms and micromobility users (based on speed similar to cycling):** 
* Cyclist 
* Bicyclist
* Bicycle
* Hand Cyclist
* Inline Skater
* Non-Motorized Scooter Rider
* Other Micromobility Device User
* Roller Skater
* Skateboarder
* Tricyclist

**Other types of vulnerable users:** 
* "Motorized Bicyclist" (MA General Law currently uses this term to mean "moped" not e-bike)
* "Motorized Scooter Rider"
* "Passenger" (Train/Trolley Passenger)
* Farm Equipment Operator
* Other
* Unknown

In [14]:
#NON_MTRST_TYPE_CL is a short string with standardized entries
#MOST_HRMFL_EVT_CL is a long string with open text
col1 = mass_crashes["NON_MTRST_TYPE_CL"]
col2 = mass_crashes["MOST_HRMFL_EVT_CL"]

mass_crashes["PEDESTRIAN"] = np.where(
    col1.str.contains("Pedestrian|Non-Motorized Wheelchair|Roller Skater", case=False, na=False) |
    col2.str.contains("Pedestrian", case=False, na=False),
    1,
    0
)

mass_crashes["CYCLIST"] = np.where(
    col1.str.contains("Cyclist|Bicyclist", case=False, na=False) |
    col2.str.contains("Cyclist|Bicyclist|Bicycle|Bike", case=False, na=False),
    1,
    0
)

mass_crashes["OTHER"] = np.where(
    mass_crashes["NON_MTRST_TYPE_CL"].str.contains("moped|other|unknown|skateboarder|Motorized Scooter Rider|passenger|Emergency Responder|Roadway Worker", case=False, na=False) |
    mass_crashes["MOST_HRMFL_EVT_CL"].str.contains("moped|other vulnerable", case=False, na=False),
    1,
    0
)

mass_crashes['SEVERITY'] = mass_crashes['CRASH_SEVERITY_DESCR'].map({
    'Fatal injury': 1,
    'Non-fatal injury': 2
}).fillna(0)


mass_crashes['INTERSTATE'] = np.where(
    mass_crashes['F_CLASS'].str.contains("Interstate", case=False, na=False),
    1,
    0
)

#TODO convert all from "Local police" to "Local", "State police" to "State", etc
mass_crashes['POLICE'] = mass_crashes['POLC_AGNCY_TYPE_DESCR']
mass_crashes['ID'] = mass_crashes['CRASH_NUMB']
mass_crashes['MUNI'] = mass_crashes['CITY_TOWN_NAME']
mass_crashes = mass_crashes.drop(columns=["CITY_TOWN_NAME", "CRASH_NUMB", "POLC_AGNCY_TYPE_DESCR"])
mass_crashes['SOURCE'] = 'MassDOT'

KeyError: 'POLC_AGNCY_TYPE_DESCR'


**Columns are attributes of each individual crash:**

* SOURCE: specifies where the crash data comes from(MASSDOT, Vision Zero, Somerville
PD and Cambridge PD)
* ID: unique identifier of each crash. Cambridge PD did not have an ID attribute so one
was created by adding CPD_ to the index number of each crash.
* MUNI: specifies the area each crash occurred(Boston, Cambridge, Brookline,
Somerville)
* DATE: the date each crash occurred in this format YEAR-MONTH-DAY
* SEVERITY: it is 1 if the crash was fatal, 2 if the crash resulted in nonfatal injuries and
blank if neither
* CRASH_TIME: time each crash occured
* POLICE: specifies if Local, State, MBTA or Campus police reported the crash
* PEDESTRIAN: 1 if pedestrian was involved in the crash, 0 if not
* CYCLIST: 1 if cyclist OR micromobility user was involved in the crash, 0 if not
* OTHER: 1 if other type of vulnerable user was involved in the crash, 0 if not
* LAT: latitude of the location where the crash occurred
* LON: longitude of the location where the crash occurred
* INTERSTATE: 1 if the crash occurred on an interstate highway, 0 if not

In [5]:
#TODO add "YEAR" below to allow for easier data checks re: 2023v issue

cols_to_move = [
    "SOURCE",
    
    "ID",

    "MUNI",

    "CRASH_DATE",

    "SEVERITY",

    "CRASH_TIME",

    "POLICE",

    "PEDESTRIAN",

    "CYCLIST",

    "OTHER",

    "LAT",

    "LON",

    "INTERSTATE"
    
]
df = mass_crashes[cols_to_move + [c for c in mass_crashes.columns if c not in cols_to_move]]
df.head()

,SOURCE,ID,MUNI,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,CYCLIST,PEDESTRIAN,OTHER,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,MassDOT,5051236,EVERETT,2022-01-01,0.0,11:34:00,Local police,0,0,0,...,NaN,Major Collector,4742067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MassDOT,5051675,EVERETT,2022-01-03,2.0,03:12:00,Local police,0,0,0,...,NaN,Principal Arterial - Other,4742121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MassDOT,5051674,EVERETT,2022-01-02,2.0,22:17:00,Local police,0,0,0,...,NaN,Local,4742122,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MassDOT,5051661,DEDHAM,2022-01-02,0.0,12:36:00,Local police,0,0,0,...,NaN,Local,4742135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MassDOT,5051578,BRAINTREE,2022-01-02,2.0,11:49:00,Local police,0,0,0,...,NaN,Minor Arterial,4742178,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# We keep only relevant columns and drop rows that don't have a date
df_short = df.iloc[:, :13]
df_short = df_short.dropna(subset=["CRASH_DATE"])

In [7]:
# We make sure numerical data is of type int or float, and strings of type str
df_short["CRASH_DATE"] = pd.to_datetime(df_short["CRASH_DATE"]).dt.date
df_short['SEVERITY'] = pd.to_numeric(df_short['SEVERITY'], errors='coerce').astype('Int64')
df_short['INTERSTATE'] = df_short['INTERSTATE'].astype(str)
df_short['ID'] = df_short['ID'].astype(str)

In [8]:
df_short.info()

<class 'pandas.DataFrame'>
RangeIndex: 219362 entries, 0 to 219361
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   SOURCE      219362 non-null  str    
 1   ID          219362 non-null  str    
 2   MUNI        219362 non-null  str    
 3   CRASH_DATE  219362 non-null  object 
 4   SEVERITY    219362 non-null  Int64  
 5   CRASH_TIME  219345 non-null  object 
 6   POLICE      219350 non-null  str    
 7   CYCLIST     219362 non-null  int64  
 8   PEDESTRIAN  219362 non-null  int64  
 9   OTHER       219362 non-null  int64  
 10  LAT         209756 non-null  float64
 11  LON         209756 non-null  float64
 12  INTERSTATE  219362 non-null  str    
dtypes: Int64(1), float64(2), int64(3), object(2), str(5)
memory usage: 22.0+ MB


In [9]:
df_short.head()

,SOURCE,ID,MUNI,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,CYCLIST,PEDESTRIAN,OTHER,LAT,LON,INTERSTATE
0,MassDOT,5051236,EVERETT,2022-01-01,0,11:34:00,Local police,0,0,0,42.408688,-71.038835,0
1,MassDOT,5051675,EVERETT,2022-01-03,2,03:12:00,Local police,0,0,0,42.406306,-71.061995,0
2,MassDOT,5051674,EVERETT,2022-01-02,2,22:17:00,Local police,0,0,0,42.420319,-71.043767,0
3,MassDOT,5051661,DEDHAM,2022-01-02,0,12:36:00,Local police,0,0,0,42.230508,-71.174294,0
4,MassDOT,5051578,BRAINTREE,2022-01-02,2,11:49:00,Local police,0,0,0,42.185967,-70.992689,0


### Renaming columns to align with Mass Crash Map format

In [10]:
renaming = {
    "LAT": "lat",
    "LON": "lng",
    "CRASH_DATE": "date",
    "CRASH_TIME": "time",
    "PEDESTRIAN": "pedestrian",
    "CYCLIST": "cyclist",
    "OTHER": "other",
    "INTERSTATE": "interstate",
    "SEVERITY": "severity",
    "ID": "id",
    "MUNI": "muni",
    "SOURCE": "source",
    "POLICE": "police"
}

df = df_short.rename(columns= renaming)

In [11]:
df.to_csv("data/boston-metro2.csv")